In [1]:
import ast
import pandas as pd

## Dataset merging & Initial Overview

In [2]:
kaggle_dataset_df = df = pd.read_csv('../data/mountain_dataset.csv', converters={'marker': ast.literal_eval})
synthetic_dataset_df = pd.read_csv('../data/synthetic_dataset.csv', converters={'marker': ast.literal_eval})

In [3]:
merged_dataset_df = pd.concat([kaggle_dataset_df, synthetic_dataset_df], ignore_index=True)

In [4]:
print("Dataset Shape:", merged_dataset_df.shape)
print("-" * 40)
merged_dataset_df.info()

Dataset Shape: (2591, 2)
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 2591 entries, 0 to 2590
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2591 non-null   str   
 1   marker  2591 non-null   object
dtypes: object(1), str(1)
memory usage: 40.6+ KB


In [5]:
merged_dataset_df.head()

,text,marker
0,A visit to a science museum for hands-on learn...,[]
1,Voice surface coach set democratic time year. ...,[]
2,Parent according maybe activity activity finis...,[]
3,A visit to a sculpture garden with intriguing ...,[]
4,The Julian Alps in Slovenia offer pristine lak...,"[(11, 15)]"


### Sentence balance

In [6]:
merged_dataset_df['mountain_count'] = merged_dataset_df['marker'].apply(len)

merged_dataset_df['contains_mountain'] = merged_dataset_df['mountain_count'] > 0

print(merged_dataset_df['contains_mountain'].value_counts())
print("-" * 40)
print(merged_dataset_df['contains_mountain'].value_counts(normalize=True) * 100)

contains_mountain
False    1359
True     1232
Name: count, dtype: int64
----------------------------------------
contains_mountain
False    52.450791
True     47.549209
Name: proportion, dtype: float64


## Entity Extraction & Frequency Analysis

In [7]:
def extract_full_entities(row):
    """
    Extracts the exact substring from the text using the character indices.
    """
    text = row['text']
    markers = row['marker']
    
    entities = []
    for start, end in markers:
        entities.append(text[start:end])
        
    return entities

In [8]:
merged_dataset_df['extracted_mountains'] = merged_dataset_df.apply(extract_full_entities, axis=1)

all_mountains = [mountain for sublist in merged_dataset_df['extracted_mountains'].tolist() for mountain in sublist]

# Calculate frequencies
mountain_freq = pd.Series(all_mountains).value_counts()

print(f"Total unique mountains: {len(mountain_freq)}")
print("-" * 40)
print("Top 15 Most Frequent Mountains:")
print(mountain_freq.head(15))

Total unique mountains: 257
----------------------------------------
Top 15 Most Frequent Mountains:
Alps                     58
Himalayas                41
Andes                    29
Appalachian Mountains    28
Mount Kilimanjaro        25
Rocky Mountains          22
Atlas Mountains          22
Ben Nevis                22
Carpathian Mountains     21
Andes Mountains          21
Sierra Nevada            21
Ural Mountains           20
Mount Fuji               19
Mount Hood               19
Mt. Rainier              18
Name: count, dtype: int64


## Creating Train & Validation datasets

In [9]:
from sklearn.model_selection import train_test_split

### Sampling train and validation data ###
train_data, val_data = train_test_split(merged_dataset_df, test_size=0.2, random_state=42, shuffle=True)
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

### Train dataset overview

In [10]:
print("Dataset Shape:", train_data.shape)
print("-" * 40)
train_data.info()

Dataset Shape: (2072, 5)
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 2072 entries, 0 to 2071
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   text                 2072 non-null   str   
 1   marker               2072 non-null   object
 2   mountain_count       2072 non-null   int64 
 3   contains_mountain    2072 non-null   bool  
 4   extracted_mountains  2072 non-null   object
dtypes: bool(1), int64(1), object(2), str(1)
memory usage: 66.9+ KB


In [11]:
### Sentence balance ###

train_data['mountain_count'] = train_data['marker'].apply(len)

train_data['contains_mountain'] = train_data['mountain_count'] > 0

print(train_data['contains_mountain'].value_counts())
print("-" * 40)
print(train_data['contains_mountain'].value_counts(normalize=True) * 100)

contains_mountain
False    1086
True      986
Name: count, dtype: int64
----------------------------------------
contains_mountain
False    52.413127
True     47.586873
Name: proportion, dtype: float64


### Validation dataset overivew

In [12]:
print("Dataset Shape:", val_data.shape)
print("-" * 40)
val_data.info()

Dataset Shape: (519, 5)
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 519 entries, 0 to 518
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   text                 519 non-null    str   
 1   marker               519 non-null    object
 2   mountain_count       519 non-null    int64 
 3   contains_mountain    519 non-null    bool  
 4   extracted_mountains  519 non-null    object
dtypes: bool(1), int64(1), object(2), str(1)
memory usage: 16.9+ KB


In [13]:
### Sentence balance ###

val_data['mountain_count'] = val_data['marker'].apply(len)

val_data['contains_mountain'] = val_data['mountain_count'] > 0

print(val_data['contains_mountain'].value_counts())
print("-" * 40)
print(val_data['contains_mountain'].value_counts(normalize=True) * 100)

contains_mountain
False    273
True     246
Name: count, dtype: int64
----------------------------------------
contains_mountain
False    52.601156
True     47.398844
Name: proportion, dtype: float64


### Saving datasets

In [14]:
train_data.to_csv("s3://qunatum-tasks-bucket/Mountains_NER/data/train_dataset.csv", index=False)
val_data.to_csv("s3://qunatum-tasks-bucket/Mountains_NER/data/val_dataset.csv", index=False)
final_dataset_df.to_csv("s3://qunatum-tasks-bucket/Mountains_NER/data/merged_dataset.csv", index=False)